# No-Comma Activation Steering

vLLM-Hook is an extensible framework that aims to allow selective access to model internals during inference.
This notebook mirrors `examples/demo_actsteer_no_comma.py`: it runs the same compact travel prompt across baseline, coefficient sweep, `adjust_rs`, and sampled cases, then summarizes comma counts.

The steering vector was derived from real Phi-3 hidden states using no-comma instruction-following examples.


### Installation

If running this from a new environment, set `RUN_INSTALL = True` in the cell below to install `vllm_hook_plugins` and the repository requirements.
The default keeps installation disabled so an already-prepared local or cluster environment can execute the notebook without reinstalling packages.


In [1]:
from pathlib import Path
import os
import subprocess
import sys

SINGLE_GPU_DEVICE = os.environ.get("NO_COMMA_CUDA_VISIBLE_DEVICES", "0")
visible = os.environ.get("CUDA_VISIBLE_DEVICES", "").strip()

if not visible:
    os.environ["CUDA_VISIBLE_DEVICES"] = SINGLE_GPU_DEVICE
    print(f"CUDA_VISIBLE_DEVICES was unset; using {SINGLE_GPU_DEVICE}.")
elif "," in visible:
    first_visible = visible.split(",", 1)[0].strip()
    os.environ["CUDA_VISIBLE_DEVICES"] = first_visible
    print(f"CUDA_VISIBLE_DEVICES had multiple GPUs ({visible}); using {first_visible}.")
else:
    print(f"CUDA_VISIBLE_DEVICES={visible}")

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
PKG_DIR = REPO_ROOT / "vllm_hook_plugins"
REQ_FILE = REPO_ROOT / "requirement.txt"
RUN_INSTALL = False

print("Notebook dir:", NOTEBOOK_DIR)
print("Repo root   :", REPO_ROOT)
print("Package dir :", PKG_DIR)
print("Req file    :", REQ_FILE)

if RUN_INSTALL:
    if REQ_FILE.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REQ_FILE)], check=True)
    else:
        print("Warning: requirement.txt not found at", REQ_FILE)
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR)], check=True)
else:
    print("Install step skipped. Set RUN_INSTALL=True if dependencies are not installed.")

plugin_src_dir = str(PKG_DIR.resolve())
if plugin_src_dir not in sys.path:
    sys.path.insert(0, plugin_src_dir)


CUDA_VISIBLE_DEVICES=0
Notebook dir: /u/yueling/autosteer/vLLM-Hook/notebooks
Repo root   : /u/yueling/autosteer/vLLM-Hook
Package dir : /u/yueling/autosteer/vLLM-Hook/vllm_hook_plugins
Req file    : /u/yueling/autosteer/vLLM-Hook/requirement.txt
Install step skipped. Set RUN_INSTALL=True if dependencies are not installed.


### Environment & Imports

Set multiprocessing and vLLM environment variables before importing vLLM, then verify that the notebook sees exactly one CUDA device.


In [2]:
import os
import multiprocessing as mp
from pathlib import Path

mp.set_start_method("spawn", force=True)
os.environ["VLLM_USE_V1"] = "1"
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

import torch
from vllm import SamplingParams
from vllm_hook_plugins import HookLLM

try:
    REPO_ROOT
except NameError:
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(REPO_ROOT)
print("Working directory:", Path.cwd())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<unset>"))
print("CUDA available:", torch.cuda.is_available())

device_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
print("GPU count:", device_count)
if torch.cuda.is_available() and device_count > 0:
    for gpu_idx in range(device_count):
        print(f"GPU {gpu_idx} name:", torch.cuda.get_device_name(gpu_idx))

if device_count != 1:
    raise RuntimeError(
        "Expected exactly one visible CUDA GPU for this notebook. "
        f"Saw {device_count}. Restart the kernel after setting "
        "CUDA_VISIBLE_DEVICES to a single GPU or launch Jupyter from an LSF "
        "GPU allocation."
    )


/u/yueling/miniconda3/envs/vllm_hook_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /u/yueling/autosteer/vLLM-Hook
CUDA_VISIBLE_DEVICES: 0
CUDA available: True
GPU count: 1
GPU 0 name: NVIDIA A100-SXM4-80GB


### Configuration

Load the same model and no-comma activation-steering config used by the Python example.


In [3]:
import json
from dataclasses import dataclass
from typing import Optional

MODEL = "microsoft/Phi-3-mini-4k-instruct"
CONFIG_PATH = "model_configs/activation_steer/Phi-3-mini-4k-instruct-no-comma.json"
cache_dir = "./cache/"

with open(CONFIG_PATH, "r") as f:
    config = json.load(f)

default_config = config["steering"]
vector_path = Path(default_config["vector_path"])
print(json.dumps(config, indent=2))
print("Config file    :", CONFIG_PATH)
print("Steering vector:", vector_path, "exists=", vector_path.exists())


{
  "steering": {
    "method": "add_vector",
    "coefficient": 40,
    "optimal_layer": 15,
    "vector_path": "steering_vectors/phi3_no_comma.pt",
    "apply_at_all_positions": true,
    "apply_to_all_tokens": true
  }
}
Config file    : model_configs/activation_steer/Phi-3-mini-4k-instruct-no-comma.json
Steering vector: steering_vectors/phi3_no_comma.pt exists= True


### Initialize `HookLLM`

Create one hook-enabled vLLM instance and reuse it for every case, matching the Python example.


In [4]:
import gc

existing_llm = globals().get("llm")
if existing_llm is not None:
    print("Shutting down existing HookLLM before creating a new one.")
    try:
        existing_llm.llm_engine.engine_core.shutdown(timeout=0)
    except Exception as exc:
        print("Existing EngineCore shutdown skipped:", repr(exc))
    try:
        existing_llm.close()
    except Exception as exc:
        print("Existing HookLLM close skipped:", repr(exc))
    del llm
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

llm = HookLLM(
    model=MODEL,
    worker_name="steer_hook_act",
    config_file=CONFIG_PATH,
    download_dir=cache_dir,
    gpu_memory_utilization=0.7,
    max_model_len=2048,
    trust_remote_code=True,
    dtype="auto",
    enforce_eager=True,
    enable_prefix_caching=True,
    enable_hook=True,
    tensor_parallel_size=1,
)


INFO 06-08 22:50:00 [utils.py:278] non-default args: {'trust_remote_code': True, 'download_dir': './cache/', 'max_model_len': 2048, 'enable_prefix_caching': True, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'worker_extension_cls': 'vllm_hook_plugins.workers.steer_activation_worker.SteerHookActWorker', 'model': 'microsoft/Phi-3-mini-4k-instruct'}


WARNING 06-08 22:50:00 [envs.py:2057] Unknown vLLM environment variable detected: VLLM_USE_V1


INFO 06-08 22:50:01 [model.py:617] Resolved architecture: Phi3ForCausalLM


INFO 06-08 22:50:01 [model.py:1752] Using max model len 2048


INFO 06-08 22:50:01 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

(EngineCore pid=1219255) INFO 06-08 22:51:54 [default_loader.py:397] Loading weights took 18.50 seconds


(EngineCore pid=1219255) INFO 06-08 22:51:55 [gpu_model_runner.py:5132] Model loading took 7.12 GiB memory and 21.391177 seconds


(EngineCore pid=1219255) INFO 06-08 22:52:00 [gpu_worker.py:466] Available KV cache memory: 47.72 GiB
(EngineCore pid=1219255) INFO 06-08 22:52:00 [kv_cache_utils.py:1733] GPU KV cache size: 129,293 tokens
(EngineCore pid=1219255) INFO 06-08 22:52:00 [kv_cache_utils.py:1734] Maximum concurrency for 2,048 tokens per request: 63.13x


(EngineCore pid=1219255) INFO 06-08 22:52:01 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=1219255) INFO 06-08 22:52:01 [core.py:309] init engine (profile, create kv cache, warmup model) took 5.76 s


(EngineCore pid=1219255) INFO 06-08 22:52:03 [vllm.py:977] Asynchronous scheduling is enabled.
(EngineCore pid=1219255) WARNING 06-08 22:52:03 [vllm.py:1033] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
(EngineCore pid=1219255) WARNING 06-08 22:52:03 [vllm.py:1058] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
(EngineCore pid=1219255) INFO 06-08 22:52:03 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])
(EngineCore pid=1219255) INFO 06-08 22:52:03 [vllm.py:1234] Cudagraph is disabled under eager mode
(EngineCore pid=1219255) INFO 06-08 22:52:03 [compilation.py:312] Enabled custom fusions: norm_quant, act_quant


### Demo Helpers

These helpers are copied from the Python example so the notebook and script exercise the same steering paths.


In [5]:
PROMPTS = {
    "travel": (
        "Write a compact five sentence itinerary for Kyoto Osaka and Nara in a "
        "Shakespearean style. Mention temples trains tea and evening walks. "
        "Do not use any commas in your response."
    ),
}


@dataclass(frozen=True)
class DemoCase:
    label: str
    prompt_key: str
    use_hook: bool
    method: Optional[str] = None
    coefficient: Optional[float] = None
    temperature: float = 0.0
    top_p: float = 1.0
    max_tokens: int = 180
    seed: Optional[int] = None


@dataclass(frozen=True)
class DemoResult:
    case: DemoCase
    text: str


def comma_count(text: str) -> int:
    return text.count(",")


def coefficient_label(case: DemoCase) -> str:
    if not case.use_hook:
        return "-"
    if case.method == "adjust_rs":
        return "n/a"
    if case.coefficient is None:
        return "json"
    return f"{case.coefficient:g}"


def apply_chat_template(llm: HookLLM, prompt: str) -> str:
    return llm.tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=False,
    )


def steering_override(default_config: dict, case: DemoCase) -> Optional[dict]:
    if not case.use_hook:
        return None

    steer = dict(default_config)
    if case.method is not None:
        steer["method"] = case.method
    if case.coefficient is not None:
        steer["coefficient"] = case.coefficient
    return steer


def sampling_params(llm: HookLLM, default_config: dict, case: DemoCase) -> SamplingParams:
    stop_token_ids = [
        token_id
        for token_id in (llm.tokenizer.eos_token_id, 32007)
        if token_id is not None
    ]
    kwargs = {
        "temperature": case.temperature,
        "top_p": case.top_p,
        "max_tokens": case.max_tokens,
        "stop_token_ids": stop_token_ids,
    }
    if case.seed is not None:
        kwargs["seed"] = case.seed

    steer = steering_override(default_config, case)
    if steer is not None:
        kwargs["extra_args"] = {"steer": steer}

    return SamplingParams(**kwargs)


def run_case(llm: HookLLM, default_config: dict, case: DemoCase) -> DemoResult:
    prompt = apply_chat_template(llm, PROMPTS[case.prompt_key])
    params = sampling_params(llm, default_config, case)
    output = llm.generate(prompt, params, use_hook=case.use_hook)[0]
    text = output.outputs[0].text.strip()

    # Prefix cache keys do not include steering params, so clear between cases.
    llm.llm_engine.reset_prefix_cache()
    return DemoResult(case=case, text=text)


def print_result(result: DemoResult) -> None:
    case = result.case
    hook = "on" if case.use_hook else "off"
    method = case.method or "none"

    print("=" * 88)
    print(
        f"[{case.label}] prompt={case.prompt_key} hook={hook} "
        f"method={method} coefficient={coefficient_label(case)} "
        f"temperature={case.temperature:g} top_p={case.top_p:g}"
    )
    print(f"Commas: {comma_count(result.text)}")
    print()
    print(result.text)


def print_summary(results: list[DemoResult]) -> None:
    print("=" * 88)
    print("Summary")
    print(
        f"{'Case':<28} {'Prompt':<8} {'Method':<10} {'Coeff':>6} "
        f"{'Temp':>5} {'Top-p':>5} {'Commas':>7}"
    )
    for result in results:
        case = result.case
        method = case.method or "none"
        print(
            f"{case.label:<28} {case.prompt_key:<8} {method:<10} "
            f"{coefficient_label(case):>6} "
            f"{case.temperature:>5g} {case.top_p:>5g} {comma_count(result.text):>7}"
        )


### Demo Cases

Run the same eight cases as `examples/demo_actsteer_no_comma.py`.


In [6]:
cases = [
    DemoCase("baseline greedy", "travel", use_hook=False),
    DemoCase(
        "weak add_vector",
        "travel",
        use_hook=True,
        method="add_vector",
        coefficient=10,
    ),
    DemoCase(
        "default add_vector",
        "travel",
        use_hook=True,
        method="add_vector",
        coefficient=default_config["coefficient"],
    ),
    DemoCase(
        "tuned add_vector",
        "travel",
        use_hook=True,
        method="add_vector",
        coefficient=55,
    ),
    DemoCase(
        "oversteered add_vector",
        "travel",
        use_hook=True,
        method="add_vector",
        coefficient=75,
    ),
    DemoCase("adjust_rs", "travel", use_hook=True, method="adjust_rs"),
    DemoCase(
        "baseline sampled",
        "travel",
        use_hook=False,
        temperature=0.7,
        top_p=0.9,
        seed=7,
    ),
    DemoCase(
        "steered sampled",
        "travel",
        use_hook=True,
        method="add_vector",
        coefficient=55,
        temperature=0.7,
        top_p=0.9,
        seed=7,
    ),
]

cases


[DemoCase(label='baseline greedy', prompt_key='travel', use_hook=False, method=None, coefficient=None, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label='weak add_vector', prompt_key='travel', use_hook=True, method='add_vector', coefficient=10, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label='default add_vector', prompt_key='travel', use_hook=True, method='add_vector', coefficient=40, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label='tuned add_vector', prompt_key='travel', use_hook=True, method='add_vector', coefficient=55, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label='oversteered add_vector', prompt_key='travel', use_hook=True, method='add_vector', coefficient=75, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label='adjust_rs', prompt_key='travel', use_hook=True, method='adjust_rs', coefficient=None, temperature=0.0, top_p=1.0, max_tokens=180, seed=None),
 DemoCase(label

### Run Demo


In [7]:
print(f"Running {len(cases)} cases with one HookLLM instance for {MODEL}.")

results = []
try:
    for case in cases:
        result = run_case(llm, default_config, case)
        results.append(result)
        print_result(result)

    print_summary(results)
finally:
    try:
        llm.llm_engine.engine_core.shutdown(timeout=0)
    except Exception as exc:
        print("EngineCore shutdown skipped:", repr(exc))
    llm.close()


Running 8 cases with one HookLLM instance for microsoft/Phi-3-mini-4k-instruct.


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

Rendering prompts: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

...

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.83s/it, est. speed input: 25.16 toks/s, output: 21.33 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.83s/it, est. speed input: 25.16 toks/s, output: 21.33 toks/s]

Processed prompts: 100%|██████████| 1/1 [00:01<00:00,  1.84s/it, est. speed input: 25.16 toks/s, output: 21.33 toks/s]

(EngineCore pid=1219255) INFO 06-08 22:52:36 [block_pool.py:482] Successfully reset prefix cache
[steered sampled] prompt=travel hook=on method=add_vector coefficient=55 temperature=0.7 top_p=0.9
Commas: 3

"Arrive Kyoto Osaka Nara. Venture forth, explore temples. Railway connects Osaka Nara. Trains ferry, tea shared, evening walks."
Summary
Case                         Prompt   Method      Coeff  Temp Top-p  Commas
baseline greedy              travel   none            -     0     1      11
weak add_vector              travel   add_vector     10     0     1       7
default add_vector           travel   add_vector     40     0     1       6
tuned add_vector             travel   add_vector     55     0     1       0
oversteered add_vector       travel   add_vector     75     0     1       0
adjust_rs                    travel   adjust_rs     n/a     0     1       7
baseline sampled             travel   none            -   0.7   0.9      12
steered sampled              travel   add_vector